# Reading the XCAP OHLCV dataset

First look at the equity price dataset built by `xcap phase1-build`. Covers how
to load it, the two joins that matter, and the traps that will silently corrupt
a backtest if you get them wrong.

**Read [`docs/DATA_RETRIEVAL.md`](../docs/DATA_RETRIEVAL.md) for why the dataset
is shaped this way.** The short version:

- `eod/` holds **raw** OHLCV — adjusted for neither splits nor dividends.
- `adjustments/` holds factors computed locally. `adjusted = close * price_factor`.
- `vendor_adjusted_close` exists **only** for reconciliation. Do not trade on it.

In [1]:
import duckdb, pandas as pd, numpy as np
from pathlib import Path

ROOT    = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
PARQUET = ROOT / "data" / "parquet"

con = duckdb.connect()
con.execute(f"SET temp_directory='{ROOT / 'data' / '_duckdb_tmp'}'")
for name, src in {
    "eod":        f"{PARQUET}/eod/**/*.parquet",
    "adj":        f"{PARQUET}/adjustments/**/*.parquet",
    "splits":     f"{PARQUET}/splits.parquet",
    "securities": f"{PARQUET}/securities.parquet",
}.items():
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{src}')")

print(con.execute("DESCRIBE eod").df().to_string(index=False))

          column_name column_type null  key default extra
          security_id     INTEGER  YES None    None  None
           api_ticker     VARCHAR  YES None    None  None
                 date        DATE  YES None    None  None
                 open      DOUBLE  YES None    None  None
                 high      DOUBLE  YES None    None  None
                  low      DOUBLE  YES None    None  None
                close      DOUBLE  YES None    None  None
vendor_adjusted_close      DOUBLE  YES None    None  None
               volume      BIGINT  YES None    None  None
                 year      BIGINT  YES None    None  None


## 1. Shape of the dataset

In [2]:
con.execute('''
    SELECT COUNT(*) AS bars, COUNT(DISTINCT security_id) AS securities,
           MIN(date) AS first_date, MAX(date) AS last_date
    FROM eod
''').df()

,bars,securities,first_date,last_date
0,58985881,31513,2000-01-03,2026-07-27


In [3]:
# Bars per year. The 2000 floor is deliberate: the vendor's delisted archive
# begins ~1997-98, so no earlier start year is survivorship-bias free.
by_year = con.execute('''
    SELECT year(date) AS year, COUNT(*) AS bars,
           COUNT(DISTINCT security_id) AS securities
    FROM eod GROUP BY 1 ORDER BY 1
''').df()
by_year.head(30)

,year,bars,securities
0,2000,1896708,8356
1,2001,1760438,7827
2,2002,1695175,7303
3,2003,1673067,7385
4,2004,1737271,7518
5,2005,1760731,7648
6,2006,1789276,7822
7,2007,1847558,8146
8,2008,1888172,8101
9,2009,1861303,7993


## 2. Survivorship — the reason this dataset exists

The universe includes securities that stopped trading. A dataset of *current*
listings would quietly exclude every company that failed, which is the single
most damaging bias in equity backtesting.

In [4]:
con.execute('''
    SELECT s.is_delisted,
           COUNT(DISTINCT e.security_id) AS securities,
           MIN(e.date) AS first_bar, MAX(e.date) AS last_bar
    FROM eod e JOIN securities s USING (security_id)
    GROUP BY 1 ORDER BY 1
''').df()

,is_delisted,securities,first_bar,last_bar
0,False,12556,2000-01-03,2026-07-27
1,True,18957,2000-01-03,2026-07-24


In [5]:
# Securities whose price history ends well before the dataset does: these are
# the names a survivorship-biased dataset would be missing entirely.
con.execute('''
    SELECT s.api_ticker, s.name, s.venue,
           MIN(e.date) AS first_bar, MAX(e.date) AS last_bar, COUNT(*) AS bars
    FROM eod e JOIN securities s USING (security_id)
    WHERE s.is_delisted
    GROUP BY 1,2,3
    HAVING MAX(e.date) < DATE '2010-01-01'
    ORDER BY bars DESC
    LIMIT 10
''').df()

,api_ticker,name,venue,first_bar,last_bar,bars
0,EPEX.US,Edge Petroleum Corp,NASDAQ,2000-01-03,2009-12-31,2515
1,APO_old.US,American Community Properties Trust,NYSE,2000-01-03,2009-12-30,2514
2,BPURQ.US,Biopure Corp,NASDAQ,2000-01-03,2009-12-23,2510
3,NRGN.US,Neurogen Corp,NASDAQ,2000-01-03,2009-12-23,2510
4,RHDCQ.US,R H Donnelley Corp,NYSE,2000-01-03,2009-12-23,2510
5,ADMGQ.US,Advanced Materials Group Inc,NASDAQ,2000-01-03,2009-12-22,2509
6,ION_old.US,Ion Media Networks Inc,NYSE,2000-01-03,2009-12-22,2509
7,NASMQ.US,North American Scientific Inc,NASDAQ,2000-01-03,2009-12-22,2509
8,MTSI1.US,Mts Medication Technologies Inc,NASDAQ,2000-01-03,2009-12-22,2509
9,IBAS.US,Ibasis Inc,NASDAQ,2000-01-03,2009-12-21,2508


## 3. Adjusted prices — the join that matters

`eod.close` is **raw**. To get a return series, multiply by `price_factor` from
`adjustments`, joined on `(security_id, date)`.

The factor is anchored so the most recent bar is exactly 1.0, and every earlier
bar carries the cumulative product of corporate actions after it.

In [6]:
px = con.execute('''
    SELECT e.date, e.close AS raw_close,
           e.close * a.price_factor AS adj_close,
           a.split_factor, a.price_factor, e.vendor_adjusted_close
    FROM eod e
    JOIN adj a USING (security_id, date)
    WHERE e.api_ticker = 'AAPL.US'
    ORDER BY e.date
''').df()

print(f"{len(px):,} bars")
print("\nAAPL around the 4-for-1 split on 2020-08-31:")
px[(px.date >= pd.Timestamp('2020-08-26')) & (px.date <= pd.Timestamp('2020-09-03'))]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

6,680 bars

AAPL around the 4-for-1 split on 2020-08-31:


,date,raw_close,adj_close,split_factor,price_factor,vendor_adjusted_close
5195,2020-08-26,506.09,126.5225,0.25,0.25,122.7235
5196,2020-08-27,500.04,125.0100,0.25,0.25,121.2564
5197,2020-08-28,499.23,124.8075,0.25,0.25,121.0600
5198,2020-08-31,129.04,129.0400,1.00,1.00,125.1654
5199,2020-09-01,134.18,134.1800,1.00,1.00,130.1511
5200,2020-09-02,131.40,131.4000,1.00,1.00,127.4545
5201,2020-09-03,120.88,120.8800,1.00,1.00,117.2504


Note the raw close drops ~4x across the split while the adjusted series is
continuous. A strategy computing returns from `raw_close` would see a fictional
-75% day.

In [7]:
raw_ret = px.set_index('date').raw_close.pct_change()
adj_ret = px.set_index('date').adj_close.pct_change()
split_day = pd.Timestamp('2020-08-31')
pd.DataFrame({
    'return from raw_close': [raw_ret.loc[split_day]],
    'return from adj_close': [adj_ret.loc[split_day]],
}, index=['2020-08-31'])

,return from raw_close,return from adj_close
2020-08-31,-0.741522,0.033912


## 4. Building a price matrix for the backtester

`BACKTEST.py` wants a date x ticker DataFrame of prices. Pivot the adjusted
series. Keep it to a small, liquid universe here — the full dataset is 59M bars
and pivoting all of it is not something to do casually.

In [8]:
UNIVERSE = ['AAPL.US','MSFT.US','JNJ.US','XOM.US','KO.US','IBM.US','GE.US','PG.US']

prices = con.execute(f'''
    SELECT e.date, e.api_ticker, e.close * a.price_factor AS px
    FROM eod e JOIN adj a USING (security_id, date)
    WHERE e.api_ticker IN ({",".join(f"'{t}'" for t in UNIVERSE)})
      AND e.date >= DATE '2010-01-01'
''').df().pivot(index='date', columns='api_ticker', values='px').sort_index()

print(prices.shape)
prices.tail(3)

(4165, 8)


api_ticker,AAPL.US,GE.US,IBM.US,JNJ.US,KO.US,MSFT.US,PG.US,XOM.US
date,,,,,,,,
2026-07-23,321.66,349.00,206.65,259.27,81.17,381.58,146.97,156.89
2026-07-24,333.02,353.73,214.19,263.40,82.25,381.70,147.41,156.94
2026-07-27,336.91,361.61,216.28,265.95,84.07,389.10,148.63,154.77


## 5. Feeding it to `BACKTEST.py`

Equal-weight, monthly rebalance, purely as an integration check that the dataset
plugs into the backtester. This is **not** a strategy.

In [9]:
import sys
sys.path.insert(0, str(Path.cwd() if Path.cwd().name == 'research' else ROOT / 'research'))
from BACKTEST import backtest

w = pd.DataFrame(1.0 / prices.shape[1], index=prices.index, columns=prices.columns)
rebal = prices.resample('ME').last().index

res = backtest(w, prices, signal_dates=list(rebal), transaction_cost=0.0005)
{k: (round(v, 4) if isinstance(v, float) else v)
 for k, v in res.items() if isinstance(v, (int, float))}

{'total_return': 5.2531,
 'ann_return': 0.1173,
 'ann_vol': 0.1555,
 'sharpe': 0.7916,
 'sortino_ratio': 0.9483,
 'max_drawdown': -0.3574,
 'avg_drawdown': -0.0394,
 'cdar': -0.1579,
 'cvar': -0.0231,
 'cvar_ann': np.float64(-0.3673),
 'downside_deviation': 0.1237,
 'expectancy': 0.0005,
 'win_rate': 0.5341,
 'avg_turnover': 0.0009,
 'ann_turnover': 0.2211,
 'total_turnover': 3.6544,
 'total_cost': 0.0087,
 'ann_borrow': 0.0,
 'total_borrow': 0.0,
 'transaction_cost': 0.0005}

## 6. Caveats you must carry into any research

1. **These are price returns, not total returns.** Dividends have not been
   downloaded yet, so `price_factor` currently equals `split_factor`. Returns are
   systematically understated for dividend payers. Check
   `data/catalog/phase1_manifest.json` -> `adjustments.factor_meaning`.

2. **Join on `security_id`, never on ticker.** Symbols are recycled after
   delisting. `api_ticker` is in `eod` for convenience only.

3. **~13% of securities with corporate actions show events dated outside their
   own price history** — spliced or recycled series. Treat those with suspicion;
   see the `spliced / recycled tickers` check in `xcap phase1-qa`.

4. **Known vendor defects**, all small but present: ~0.04% of bars have
   non-positive prices, ~0.01% violate `low <= {open,close} <= high`. They are
   flagged rather than silently repaired, because dropping vs winsorising is a
   strategy decision.

5. **The universe is deliberately not filtered by liquidity or price.** It
   includes sub-penny microcaps. Apply your own eligibility screen.

In [10]:
import json
m = json.load(open(ROOT / 'data' / 'catalog' / 'phase1_manifest.json'))
print('start_date      :', m['start_date'])
print('skipped blocks  :', m.get('skipped', {}))
a = json.load(open(ROOT / 'data' / 'catalog' / 'progress.json'))
print('universe        :', f"{a['universe_size']:,} securities")

start_date      : 2000-01-01
skipped blocks  : {'dividends': '300/32,525 securities resolved - block incomplete, not built'}
universe        : 32,525 securities
